In [ ]:
import pandas as pd
import numpy as np
from scipy.interpolate import PchipInterpolator
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M')
df = df.sort_values('datetime').reset_index(drop=True)
N = len(df)

def get_strike(col):
    return int(col.replace('NIFTY27JAN26', '')[:-2])

ce_cols = sorted([c for c in df.columns if 'CE' in c], key=get_strike)
pe_cols = sorted([c for c in df.columns if 'PE' in c], key=get_strike)
iv_cols = ce_cols + pe_cols
ce_strikes = np.array([get_strike(c) for c in ce_cols])
pe_strikes = np.array([get_strike(c) for c in pe_cols])
spot_vals = df['underlying_price'].values

# Detect expiry dates (last 2 dates)
dates = df['datetime'].dt.date
all_dates = sorted(dates.unique())
expiry_dates = set(all_dates[-2:])
expiry_mask = dates.isin(expiry_dates).values

# Fine-tuned parameters
SM = 0.60
EW = 0.70
ALPHA = 0.15

filled_cross = df[iv_cols].copy()
for ti in range(N):
    spot = spot_vals[ti]
    is_expiry = expiry_mask[ti]

    for cols_grp, strikes_arr in [(ce_cols, ce_strikes), (pe_cols, pe_strikes)]:
        row_vals = filled_cross.loc[ti, cols_grp].values.astype(float)
        obs, miss = ~np.isnan(row_vals), np.isnan(row_vals)
        if obs.sum() < 2 or miss.sum() == 0:
            continue

        moneyness = np.log(strikes_arr / spot)
        obs_x = moneyness[obs]
        obs_y = row_vals[obs]
        sort_i = np.argsort(obs_x)
        obs_xs = obs_x[sort_i]
        obs_ys = obs_y[sort_i]

        for mi in np.where(miss)[0]:
            tgt_m = moneyness[mi]
            left_mask = obs_xs < tgt_m
            right_mask = obs_xs > tgt_m
            has_left = left_mask.any()
            has_right = right_mask.any()

            if is_expiry:
                # EXPIRY: use local linear between immediate neighbors
                if has_left and has_right:
                    lx, ly = obs_xs[left_mask][-1], obs_ys[left_mask][-1]
                    rx, ry = obs_xs[right_mask][0], obs_ys[right_mask][0]
                    t = (tgt_m - lx) / (rx - lx) if abs(rx - lx) > 1e-10 else 0.5
                    pred = ly + t * (ry - ly)
                elif has_left:
                    lx2, ly2 = obs_xs[left_mask][-1], obs_ys[left_mask][-1]
                    if left_mask.sum() >= 2:
                        lx1, ly1 = obs_xs[left_mask][-2], obs_ys[left_mask][-2]
                        slope = (ly2 - ly1) / (lx2 - lx1 + 1e-10)
                        slope = min(slope, abs(ly2 / (lx2 + 1e-10)) * 1.5)
                    else:
                        slope = 0
                    pred = ly2 + slope * (tgt_m - lx2)
                else:
                    rx2, ry2 = obs_xs[right_mask][0], obs_ys[right_mask][0]
                    if right_mask.sum() >= 2:
                        rx1, ry1 = obs_xs[right_mask][1], obs_ys[right_mask][1]
                        slope = (ry1 - ry2) / (rx1 - rx2 + 1e-10)
                        slope = min(slope, abs(ry2 / (abs(rx2) + 1e-10)) * 1.5)
                    else:
                        slope = 0
                    pred = ry2 + slope * (tgt_m - rx2)
            else:
                # NORMAL: weighted poly with tuned parameters
                sigma_w = SM * max((obs_x.max() - obs_x.min()) / 2, 0.01)
                weights = np.maximum(np.exp(-0.5 * ((obs_x - tgt_m) / sigma_w) ** 2), 0.05)
                is_extrap = not has_left or not has_right
                try:
                    coeffs = np.polyfit(obs_x, obs_y, deg=min(3, obs.sum()-1), w=weights)
                    poly_pred = float(np.poly1d(coeffs)(tgt_m))
                except:
                    poly_pred = float(np.interp(tgt_m, obs_x, obs_y))

                if not is_extrap:
                    pred = poly_pred
                else:
                    if not has_left:
                        si = np.argsort(obs_x)[:2]
                    else:
                        si = np.argsort(obs_x)[-2:]
                    x1, x2 = obs_x[si[0]], obs_x[si[1]]
                    y1, y2 = obs_y[si[0]], obs_y[si[1]]
                    lin = y1 + (y2-y1)/(x2-x1+1e-10)*(tgt_m-x1)
                    pred = EW*poly_pred + (1-EW)*lin

            filled_cross.loc[ti, cols_grp[mi]] = max(pred, 0.005)

filled_cross = filled_cross.ffill().bfill()

filled_temp = df[iv_cols].copy()
for col in iv_cols:
    series = df[col].copy()
    obs_mask = ~series.isna()
    ox = np.where(obs_mask)[0]
    oy = series[obs_mask].values
    miss_x = np.where(~obs_mask)[0]
    if len(miss_x) == 0 or len(ox) < 2:
        continue
    pi = PchipInterpolator(ox, oy, extrapolate=True)
    filled_temp.loc[miss_x, col] = np.clip(pi(miss_x), 0.005, None)
filled_temp = filled_temp.ffill().bfill()

final_df = df[['datetime', 'underlying_price'] + iv_cols].copy()
miss_orig = df[iv_cols].isna()
for col in iv_cols:
    miss = miss_orig[col]
    final_df.loc[miss, col] = ((1-ALPHA) * filled_cross.loc[miss, col].values
                               + ALPHA * filled_temp.loc[miss, col].values)
for col in iv_cols:
    final_df[col] = final_df[col].clip(lower=0.005)

assert final_df[iv_cols].isnull().sum().sum() == 0
print(f"IV range: {final_df[iv_cols].min().min():.4f} - {final_df[iv_cols].max().max():.4f}")

final_df['datetime'] = df['datetime'].dt.strftime('%d-%m-%Y %H:%M')
final_df[['datetime', 'underlying_price'] + iv_cols].to_csv('../outputs/2_submissions/filled_dataset.csv', index=False)

original = pd.read_csv('../data/raw/dataset.csv')
filled = pd.read_csv('../outputs/2_submissions/filled_dataset.csv')
rows = []
for col in [c for c in original.columns if c not in ('datetime', 'underlying_price')]:
    for idx in original.index[original[col].isna()]:
        rows.append({'id': f"{original.loc[idx, 'datetime']}||{col}",
                     'value': filled.loc[idx, col]})
pd.DataFrame(rows, columns=['id', 'value']).sort_values(
    'id').reset_index(drop=True).to_csv('../outputs/2_submissions/submission.csv', index=False)
print("Done!")

IV range: 0.0168 - 5.4970
Done!


In [19]:
submission_df = pd.read_csv('../outputs/2_submissions/submission.csv')
submission_df.shape

(5460, 2)

In [29]:
from scipy.interpolate import interp1d
from sklearn.metrics import mean_squared_error
# Last 2 dates treated as expiry dates
# --------------------------------------------------

# --------------------------------------------------
# Linear interpolation predictor
# --------------------------------------------------
def predict_linear_interp(mask_rows, df_context):

    preds = []

    for _, row in mask_rows.iterrows():

        ts_data = df_context[
            (df_context['datetime'] == row['datetime']) &
            (df_context['option_type'] == row['option_type']) &
            (df_context['iv'].notna())
        ].sort_values('strike')

        if len(ts_data) >= 2:

            try:

                f = interp1d(
                    ts_data['strike'].values,
                    ts_data['iv'].values,
                    kind='linear',
                    bounds_error=False,
                    fill_value=(
                        ts_data['iv'].iloc[0],
                        ts_data['iv'].iloc[-1]
                    )
                )

                preds.append(float(f(row['strike'])))

            except:
                preds.append(np.nan)

        else:
            preds.append(np.nan)

    return np.array(preds)


# --------------------------------------------------
# Evaluate only expiry dates
# --------------------------------------------------

df = pd.read_csv('../data/processed/dataset_long_pruned.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
dates = sorted(df['datetime'].dt.date.unique())
expiry_df = df[df['datetime'].dt.date == dates[-1]].copy()

print(f"Rows on expiry dates: {len(expiry_df)}")

np.random.seed(42)

available_rows = expiry_df[expiry_df['iv'].notna()].index

hidden_rows = np.random.choice(
    available_rows,
    size=int(0.20 * len(available_rows)),
    replace=False
)

ground_truth = expiry_df.loc[hidden_rows, 'iv'].values

expiry_masked = expiry_df.copy()
expiry_masked.loc[hidden_rows, 'iv'] = np.nan

mask_rows = expiry_masked.loc[hidden_rows]

preds = predict_linear_interp(
    mask_rows=mask_rows,
    df_context=expiry_masked
)

valid_mask = ~np.isnan(preds)

mse = mean_squared_error(
    ground_truth[valid_mask],
    preds[valid_mask]
)

rmse = np.sqrt(mse)

print(f"\nHidden rows       : {len(hidden_rows)}")
print(f"Predictions made  : {valid_mask.sum()}")
print(f"MSE               : {mse:.8f}")
print(f"RMSE              : {rmse:.8f}")
coverage = valid_mask.mean()

print(f"Coverage          : {coverage:.2%}")

Rows on expiry dates: 2100

Hidden rows       : 338
Predictions made  : 338
MSE               : 0.01365653
RMSE              : 0.11686116
Coverage          : 100.00%


In [ ]:
"""
Combined IV Surface Imputation Pipeline
----------------------------------------
Strategy:
  - NON-EXPIRY days  → MLP
  - EXPIRY days      → Cross-sectional poly + PCHIP blend
"""

import pandas as pd
import numpy as np
import warnings
from scipy.interpolate import PchipInterpolator
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 0. CONFIG
# ─────────────────────────────────────────────
RAW_CSV      = '../data/raw/dataset.csv'
OUTPUT_CSV   = '../outputs/2_submissions/filled_dataset.csv'
SUBMIT_CSV   = '../outputs/2_submissions/submission.csv'

SM    = 0.60
EW    = 0.70
ALPHA = 0.15   # weight of PCHIP vs cross-sectional poly

# MLP training params
EPOCHS     = 500
BATCH_SIZE = 256
PATIENCE   = 40
LR         = 5e-4
WEIGHT_DECAY = 1e-5

FEATURE_COLS = [
    'iv_roll_mean_10', 'iv_roll_mean_5',
    'iv_neighbor_mean', 'wide_iv_neighbor_mean',
    'strike',
    'iv_neighbor_+1', 'iv_neighbor_-1',
    'iv_neighbor_+2', 'iv_neighbor_-2',
    'dist_from_atm_pct', 'moneyness', 'log_moneyness'
]

FOLDS = [
    {'name': 'Fold 1', 'train': slice(None, 9),  'val': slice(9, 11)},
    {'name': 'Fold 2', 'train': slice(None, 10), 'val': slice(10, 12)},
]


# ─────────────────────────────────────────────
# 1. LOAD RAW DATA
# ─────────────────────────────────────────────
def load_raw():
    df = pd.read_csv(RAW_CSV)
    df['datetime'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M')
    df = df.sort_values('datetime').reset_index(drop=True)
    return df


def get_strike(col):
    return int(col.replace('NIFTY27JAN26', '')[:-2])


def get_option_cols(df):
    ce_cols = sorted([c for c in df.columns if 'CE' in c], key=get_strike)
    pe_cols = sorted([c for c in df.columns if 'PE' in c], key=get_strike)
    return ce_cols, pe_cols

def friends_fill_wide(df, ce_cols, pe_cols):
    """Cross-sectional poly (non-expiry) / linear (expiry) + PCHIP blend."""
    iv_cols   = ce_cols + pe_cols
    ce_strikes = np.array([get_strike(c) for c in ce_cols])
    pe_strikes = np.array([get_strike(c) for c in pe_cols])
    spot_vals  = df['underlying_price'].values
    N          = len(df)

    dates       = df['datetime'].dt.date
    all_dates   = sorted(dates.unique())
    expiry_dates = set(all_dates[-2:])          # last 2 calendar dates
    expiry_mask  = dates.isin(expiry_dates).values

    # --- Pass 1: cross-sectional fill ---
    filled_cross = df[iv_cols].copy()
    for ti in range(N):
        spot      = spot_vals[ti]
        is_expiry = expiry_mask[ti]

        for cols_grp, strikes_arr in [(ce_cols, ce_strikes), (pe_cols, pe_strikes)]:
            row_vals = filled_cross.loc[ti, cols_grp].values.astype(float)
            obs  = ~np.isnan(row_vals)
            miss = np.isnan(row_vals)
            if obs.sum() < 2 or miss.sum() == 0:
                continue

            moneyness = np.log(strikes_arr / spot)
            obs_x = moneyness[obs];  obs_y = row_vals[obs]
            sort_i = np.argsort(obs_x)
            obs_xs = obs_x[sort_i];  obs_ys = obs_y[sort_i]

            for mi in np.where(miss)[0]:
                tgt_m      = moneyness[mi]
                left_mask  = obs_xs < tgt_m
                right_mask = obs_xs > tgt_m
                has_left   = left_mask.any()
                has_right  = right_mask.any()

                if is_expiry:
                    if has_left and has_right:
                        lx, ly = obs_xs[left_mask][-1], obs_ys[left_mask][-1]
                        rx, ry = obs_xs[right_mask][0], obs_ys[right_mask][0]
                        t    = (tgt_m - lx) / (rx - lx) if abs(rx - lx) > 1e-10 else 0.5
                        pred = ly + t * (ry - ly)
                    elif has_left:
                        lx2, ly2 = obs_xs[left_mask][-1], obs_ys[left_mask][-1]
                        if left_mask.sum() >= 2:
                            lx1, ly1 = obs_xs[left_mask][-2], obs_ys[left_mask][-2]
                            slope = (ly2 - ly1) / (lx2 - lx1 + 1e-10)
                            slope = min(slope, abs(ly2 / (lx2 + 1e-10)) * 1.5)
                        else:
                            slope = 0
                        pred = ly2 + slope * (tgt_m - lx2)
                    else:
                        rx2, ry2 = obs_xs[right_mask][0], obs_ys[right_mask][0]
                        if right_mask.sum() >= 2:
                            rx1, ry1 = obs_xs[right_mask][1], obs_ys[right_mask][1]
                            slope = (ry1 - ry2) / (rx1 - rx2 + 1e-10)
                            slope = min(slope, abs(ry2 / (abs(rx2) + 1e-10)) * 1.5)
                        else:
                            slope = 0
                        pred = ry2 + slope * (tgt_m - rx2)
                else:
                    obs_x_all = moneyness[obs]
                    sigma_w   = SM * max((obs_x_all.max() - obs_x_all.min()) / 2, 0.01)
                    weights   = np.maximum(
                        np.exp(-0.5 * ((obs_x_all - tgt_m) / sigma_w) ** 2), 0.05)
                    is_extrap = not has_left or not has_right
                    try:
                        coeffs    = np.polyfit(obs_x_all, obs_y, deg=min(3, obs.sum()-1), w=weights)
                        poly_pred = float(np.poly1d(coeffs)(tgt_m))
                    except Exception:
                        poly_pred = float(np.interp(tgt_m, obs_xs, obs_ys))

                    if not is_extrap:
                        pred = poly_pred
                    else:
                        si       = np.argsort(obs_x_all)[:2] if not has_left else np.argsort(obs_x_all)[-2:]
                        x1, x2  = obs_x_all[si[0]], obs_x_all[si[1]]
                        y1, y2  = obs_y[si[0]], obs_y[si[1]]
                        lin     = y1 + (y2-y1)/(x2-x1+1e-10)*(tgt_m-x1)
                        pred    = EW*poly_pred + (1-EW)*lin

                filled_cross.loc[ti, cols_grp[mi]] = max(pred, 0.005)

    filled_cross = filled_cross.ffill().bfill()

    # --- Pass 2: PCHIP temporal fill ---
    filled_temp = df[iv_cols].copy()
    for col in iv_cols:
        series   = df[col].copy()
        obs_mask = ~series.isna()
        ox       = np.where(obs_mask)[0]
        oy       = series[obs_mask].values
        miss_x   = np.where(~obs_mask)[0]
        if len(miss_x) == 0 or len(ox) < 2:
            continue
        pi = PchipInterpolator(ox, oy, extrapolate=True)
        filled_temp.loc[miss_x, col] = np.clip(pi(miss_x), 0.005, None)
    filled_temp = filled_temp.ffill().bfill()

    # --- Blend ---
    final = df[iv_cols].copy()
    miss_orig = df[iv_cols].isna()
    for col in iv_cols:
        miss = miss_orig[col]
        final.loc[miss, col] = (
            (1 - ALPHA) * filled_cross.loc[miss, col].values
            + ALPHA     * filled_temp.loc[miss, col].values
        )
    final = final.clip(lower=0.005)
    return final, expiry_mask   # wide-format filled values


# ─────────────────────────────────────────────
# 3. MLP (your architecture)
# ─────────────────────────────────────────────
class IVSurfaceMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=None, dropout=0.1):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [128, 64, 32]
        layers = []; prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(X_train, y_train, X_val, y_val,
              epochs=EPOCHS, batch_size=BATCH_SIZE, patience=PATIENCE):
    scaler      = StandardScaler()
    X_train_trf = scaler.fit_transform(X_train)
    X_val_trf   = scaler.transform(X_val)

    Xtt = torch.tensor(X_train_trf, dtype=torch.float32)
    ytt = torch.tensor(y_train,     dtype=torch.float32)
    Xvt = torch.tensor(X_val_trf,   dtype=torch.float32)
    yvt = torch.tensor(y_val,       dtype=torch.float32)

    loader  = DataLoader(TensorDataset(Xtt, ytt), batch_size=batch_size, shuffle=True)
    model   = IVSurfaceMLP(X_train.shape[1])
    opt     = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.MSELoss()

    best_loss, best_state, no_imp = float('inf'), None, 0
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train(); epoch_loss = 0
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item()
        sched.step()
        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(Xvt), yvt).item()
        train_losses.append(epoch_loss / len(loader))
        val_losses.append(vl)
        if vl < best_loss:
            best_loss  = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp     = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    model.load_state_dict(best_state)
    return model, scaler, epoch - patience, train_losses, val_losses


def predict_mlp(model, scaler, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(scaler.transform(X), dtype=torch.float32)).numpy()


# ─────────────────────────────────────────────
# 4. FEATURE ENGINEERING for MLP  (long format)
# ─────────────────────────────────────────────
def build_long_features(df_wide, ce_cols, pe_cols):
    """
    Convert wide IV dataframe to long format and engineer features.
    Works on whatever rows df_wide contains.
    """
    iv_cols = ce_cols + pe_cols
    rows = []
    for _, row in df_wide.iterrows():
        spot = row['underlying_price']
        for opt_type, cols_grp in [('CE', ce_cols), ('PE', pe_cols)]:
            strikes = [get_strike(c) for c in cols_grp]
            ivs     = [row[c] for c in cols_grp]
            for i, (strike, iv) in enumerate(zip(strikes, ivs)):
                rows.append({
                    'datetime':    row['datetime'],
                    'strike':      strike,
                    'opt_type':    opt_type,
                    'iv':          iv,
                    'spot':        spot,
                    'moneyness':   strike / spot,
                    'log_moneyness': np.log(strike / spot),
                    'dist_from_atm_pct': abs(strike - spot) / spot * 100,
                    'col_idx':     i,
                    'cols_grp':    cols_grp,
                    'ivs_list':    ivs,
                })
    long_df = pd.DataFrame(rows)

    # neighbor IVs (same row, adjacent strikes)
    def neighbor_iv(row, offset):
        idx = row['col_idx'] + offset
        ivs = row['ivs_list']
        return ivs[idx] if 0 <= idx < len(ivs) else np.nan

    long_df['iv_neighbor_+1'] = long_df.apply(lambda r: neighbor_iv(r, +1), axis=1)
    long_df['iv_neighbor_-1'] = long_df.apply(lambda r: neighbor_iv(r, -1), axis=1)
    long_df['iv_neighbor_+2'] = long_df.apply(lambda r: neighbor_iv(r, +2), axis=1)
    long_df['iv_neighbor_-2'] = long_df.apply(lambda r: neighbor_iv(r, -2), axis=1)

    def neighbor_mean_narrow(row):
        ivs = row['ivs_list']; idx = row['col_idx']
        ns  = [ivs[i] for i in [idx-1, idx+1] if 0 <= i < len(ivs)]
        return np.nanmean(ns) if ns else np.nan

    def neighbor_mean_wide(row):
        ivs = row['ivs_list']; idx = row['col_idx']
        ns  = [ivs[i] for i in [idx-2, idx-1, idx+1, idx+2] if 0 <= i < len(ivs)]
        return np.nanmean(ns) if ns else np.nan

    long_df['iv_neighbor_mean']      = long_df.apply(neighbor_mean_narrow, axis=1)
    long_df['wide_iv_neighbor_mean'] = long_df.apply(neighbor_mean_wide, axis=1)

    # temporal rolling means per strike
    long_df = long_df.sort_values(['strike', 'opt_type', 'datetime']).reset_index(drop=True)
    long_df['iv_roll_mean_5']  = long_df.groupby(['strike','opt_type'])['iv'].transform(
        lambda s: s.shift(1).rolling(5,  min_periods=1).mean())
    long_df['iv_roll_mean_10'] = long_df.groupby(['strike','opt_type'])['iv'].transform(
        lambda s: s.shift(1).rolling(10, min_periods=1).mean())

    long_df = long_df.fillna(0)
    return long_df


# ─────────────────────────────────────────────
# 5. MAKE HOLDOUT (mirrors your friend's eval)
# ─────────────────────────────────────────────
def make_holdout(val_df):
    np.random.seed(42)
    available_rows = val_df[val_df['iv'].notna()].index
    hidden_rows    = np.random.choice(available_rows,
                                      size=int(0.2 * len(available_rows)),
                                      replace=False)
    ground_truth   = val_df.loc[hidden_rows, 'iv'].values
    val_masked     = val_df.copy()
    val_masked.loc[hidden_rows, 'iv'] = np.nan
    return val_masked, hidden_rows, ground_truth


# ─────────────────────────────────────────────
# 6. MAIN
# ─────────────────────────────────────────────
def main():
    print("Loading data...")
    df_raw   = load_raw()
    ce_cols, pe_cols = get_option_cols(df_raw)
    iv_cols  = ce_cols + pe_cols

    dates      = df_raw['datetime'].dt.date
    all_dates  = sorted(dates.unique())
    expiry_dates = set(all_dates[-2:])          # last 2 unique calendar dates
    expiry_mask  = dates.isin(expiry_dates).values

    print(f"Total rows: {len(df_raw)}  |  Expiry rows: {expiry_mask.sum()}")

    # ── A. Run friend's full-pipeline to get expiry-day fills ──────────────
    print("\nRunning friend's cross-sectional fill for expiry days...")
    friends_filled, _ = friends_fill_wide(df_raw, ce_cols, pe_cols)

    # ── B. Build long-format for MLP on NON-EXPIRY rows only ───────────────
    print("Building long-format features for MLP (non-expiry)...")
    df_non_expiry = df_raw[~expiry_mask].copy().reset_index(drop=True)
    long_df = build_long_features(df_non_expiry, ce_cols, pe_cols)

    # ── C. Cross-val loop (non-expiry MLP) ─────────────────────────────────
    non_expiry_dates = [d for d in all_dates if d not in expiry_dates]
    FOLDS_NE = [
        {'name': 'Fold 1', 'train': non_expiry_dates[:9],  'val': non_expiry_dates[9:11]},
        {'name': 'Fold 2', 'train': non_expiry_dates[:10], 'val': non_expiry_dates[10:12]},
    ]

    fold_mses = []
    for fold in FOLDS_NE:
        train_df = long_df[long_df['datetime'].dt.date.isin(fold['train'])].copy()
        val_df   = long_df[long_df['datetime'].dt.date.isin(fold['val'])].copy()

        val_masked, hidden_rows, ground_truth = make_holdout(val_df)

        train_obs = train_df[train_df['iv'].notna()]
        X_all     = train_obs[FEATURE_COLS].fillna(0).values
        y_all     = train_obs['iv'].values

        split = int(0.85 * len(X_all))
        model, scaler, best_ep, tl, vl = train_mlp(
            X_all[:split], y_all[:split],
            X_all[split:], y_all[split:])

        X_pred = val_masked.loc[hidden_rows, FEATURE_COLS].fillna(0).values
        preds  = predict_mlp(model, scaler, X_pred)
        mse    = mean_squared_error(ground_truth, preds)
        fold_mses.append(mse)
        print(f"  {fold['name']}: MSE={mse:.6f}  best_epoch={best_ep}")

    mlp_mse = np.mean(fold_mses)
    print(f"MLP (non-expiry) CV MSE : {mlp_mse:.6f}")

    # ── D. Train FINAL MLP on ALL non-expiry observed data ─────────────────
    print("\nTraining final MLP on all non-expiry data...")
    obs_all = long_df[long_df['iv'].notna()]
    X_final = obs_all[FEATURE_COLS].fillna(0).values
    y_final = obs_all['iv'].values
    split   = int(0.9 * len(X_final))
    final_model, final_scaler, _, _, _ = train_mlp(
        X_final[:split], y_final[:split],
        X_final[split:], y_final[split:])

    # ── E. Fill missing non-expiry cells with MLP ──────────────────────────
    print("Filling non-expiry missing values with MLP...")
    final_wide = df_raw[['datetime','underlying_price'] + iv_cols].copy()
    miss_orig  = df_raw[iv_cols].isna()

    # Rebuild long-format for the FULL dataset (to get features for missing cells too)
    long_full = build_long_features(df_raw, ce_cols, pe_cols)
    # Map (datetime, strike, opt_type) → MLP prediction
    long_miss = long_full[long_full['iv'].isna() & ~long_full['datetime'].dt.date.isin(expiry_dates)]
    if len(long_miss) > 0:
        X_miss = long_miss[FEATURE_COLS].fillna(0).values
        preds_miss = predict_mlp(final_model, final_scaler, X_miss).clip(min=0.005)
        long_miss = long_miss.copy()
        long_miss['iv_pred'] = preds_miss

        # Write MLP predictions back to wide format
        for _, r in long_miss.iterrows():
            dt      = r['datetime']
            strike  = int(r['strike'])
            otype   = r['opt_type']
            col_name = f"NIFTY27JAN26{strike}{otype}"
            if col_name in iv_cols:
                row_idx = final_wide[final_wide['datetime'] == dt].index
                if len(row_idx) > 0 and pd.isna(final_wide.loc[row_idx[0], col_name]):
                    final_wide.loc[row_idx[0], col_name] = r['iv_pred']

    # ── F. Overwrite EXPIRY rows with friend's fills ────────────────────────
    print("Applying friend's fills to expiry rows...")
    expiry_row_idx = df_raw.index[expiry_mask]
    for col in iv_cols:
        miss_expiry = miss_orig[col] & expiry_mask
        miss_idx    = df_raw.index[miss_expiry]
        if len(miss_idx):
            final_wide.loc[miss_idx, col] = friends_filled.loc[miss_idx, col].values

    # ── G. Safety ffill/bfill for any remaining NaN ──────────────────────────
    still_missing = final_wide[iv_cols].isna().sum().sum()
    if still_missing:
        print(f"  {still_missing} cells still NaN after both methods — applying ffill/bfill")
        final_wide[iv_cols] = final_wide[iv_cols].ffill().bfill()

    final_wide[iv_cols] = final_wide[iv_cols].clip(lower=0.005)

    assert final_wide[iv_cols].isnull().sum().sum() == 0, "NaNs remain!"
    print(f"IV range: {final_wide[iv_cols].min().min():.4f} – {final_wide[iv_cols].max().max():.4f}")

    # ── H. Save outputs ────────────────────────────────────────────────────
    final_wide['datetime'] = df_raw['datetime'].dt.strftime('%d-%m-%Y %H:%M')
    final_wide[['datetime','underlying_price'] + iv_cols].to_csv(OUTPUT_CSV, index=False)

    original = pd.read_csv(RAW_CSV)
    filled   = pd.read_csv(OUTPUT_CSV)
    rows_out = []
    for col in [c for c in original.columns if c not in ('datetime','underlying_price')]:
        for idx in original.index[original[col].isna()]:
            rows_out.append({'id':    f"{original.loc[idx,'datetime']}||{col}",
                             'value': filled.loc[idx, col]})
    (pd.DataFrame(rows_out, columns=['id','value'])
       .sort_values('id').reset_index(drop=True)
       .to_csv(SUBMIT_CSV, index=False))

    print(f"\nDone! submission → {SUBMIT_CSV}")
    print(f"Estimated overall MSE (if non-expiry MLP ≈ {mlp_mse:.6f}): "
          f"target ≤ 0.000035")


if __name__ == '__main__':
    main()

Loading data...
Total rows: 975  |  Expiry rows: 150

Running friend's cross-sectional fill for expiry days...
Building long-format features for MLP (non-expiry)...
  Fold 1: MSE=0.004156  best_epoch=87
  Fold 2: MSE=0.004313  best_epoch=58
MLP (non-expiry) CV MSE : 0.004235

Training final MLP on all non-expiry data...
Filling non-expiry missing values with MLP...
Applying friend's fills to expiry rows...
  4642 cells still NaN after both methods — applying ffill/bfill
IV range: 0.0168 – 5.4970

Done! submission → ../outputs/2_submissions/submission.csv
Estimated overall MSE (if non-expiry MLP ≈ 0.004235): target ≤ 0.000035


In [39]:
"""
Strategy:
  - NON-EXPIRY days  → MLP
  - EXPIRY day       → Cross-sectional linear interpolation + PCHIP blend
"""

import pandas as pd
import numpy as np
import warnings
from scipy.interpolate import PchipInterpolator
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 0. CONFIG
# ─────────────────────────────────────────────
RAW_CSV    = '../data/raw/dataset.csv'
OUTPUT_CSV = '../outputs/2_submissions/filled_dataset.csv'
SUBMIT_CSV = '../outputs/2_submissions/submission.csv'

SM    = 0.60
EW    = 0.70
ALPHA = 0.15   # weight of PCHIP vs cross-sectional in expiry blend

# MLP training params
EPOCHS       = 500
BATCH_SIZE   = 256
PATIENCE     = 40
LR           = 5e-4
WEIGHT_DECAY = 1e-5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FEATURE_COLS = [
    'iv_roll_mean_10', 'iv_roll_mean_5',
    'iv_neighbor_mean', 'wide_iv_neighbor_mean',
    'strike',
    'iv_neighbor_+1', 'iv_neighbor_-1',
    'iv_neighbor_+2', 'iv_neighbor_-2',
    'dist_from_atm_pct', 'moneyness', 'log_moneyness',
    'is_ce',   # fix: MLP needs to distinguish CE vs PE
]


# ─────────────────────────────────────────────
# 1. LOAD RAW DATA
# ─────────────────────────────────────────────
def load_raw():
    df = pd.read_csv(RAW_CSV)
    df['datetime'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M')
    df = df.sort_values('datetime').reset_index(drop=True)
    return df


def get_strike(col):
    return int(col.replace('NIFTY27JAN26', '')[:-2])


def get_option_cols(df):
    ce_cols = sorted([c for c in df.columns if c.endswith('CE')], key=get_strike)
    pe_cols = sorted([c for c in df.columns if c.endswith('PE')], key=get_strike)
    return ce_cols, pe_cols


# ─────────────────────────────────────────────
# 2. EXPIRY DAY FILL  (cross-sectional linear + PCHIP blend)
# ─────────────────────────────────────────────
def fill_wide(df, ce_cols, pe_cols):
    """
    Cross-sectional linear interpolation + PCHIP temporal blend
    for expiry day rows only.
    """
    # fix: always work on a clean 0-based RangeIndex so .loc[ti, ...] is safe
    df = df.reset_index(drop=True)

    iv_cols    = ce_cols + pe_cols
    ce_strikes = np.array([get_strike(c) for c in ce_cols])
    pe_strikes = np.array([get_strike(c) for c in pe_cols])
    spot_vals  = df['underlying_price'].values
    N          = len(df)

    dates        = df['datetime'].dt.date
    all_dates    = sorted(dates.unique())
    expiry_date  = all_dates[-1]                         # last day only
    expiry_mask  = (dates == expiry_date).values

    # ── Pass 1: cross-sectional fill (expiry rows only) ───────────────────
    filled_cross = df[iv_cols].copy()

    for ti in range(N):
        if not expiry_mask[ti]:
            continue

        spot = spot_vals[ti]

        for cols_grp, strikes_arr in [(ce_cols, ce_strikes), (pe_cols, pe_strikes)]:
            row_vals = filled_cross.loc[ti, cols_grp].values.astype(float)
            obs  = ~np.isnan(row_vals)
            miss = np.isnan(row_vals)
            if obs.sum() < 2 or miss.sum() == 0:
                continue

            moneyness = np.log(strikes_arr / spot)
            obs_x  = moneyness[obs]
            obs_y  = row_vals[obs]
            sort_i = np.argsort(obs_x)
            obs_xs = obs_x[sort_i]
            obs_ys = obs_y[sort_i]

            for mi in np.where(miss)[0]:
                tgt_m      = moneyness[mi]
                left_mask  = obs_xs < tgt_m
                right_mask = obs_xs > tgt_m
                has_left   = left_mask.any()
                has_right  = right_mask.any()

                if has_left and has_right:
                    lx, ly = obs_xs[left_mask][-1], obs_ys[left_mask][-1]
                    rx, ry = obs_xs[right_mask][0],  obs_ys[right_mask][0]
                    t      = (tgt_m - lx) / (rx - lx) if abs(rx - lx) > 1e-10 else 0.5
                    pred   = ly + t * (ry - ly)

                elif has_left:
                    lx2, ly2 = obs_xs[left_mask][-1], obs_ys[left_mask][-1]
                    if left_mask.sum() >= 2:
                        lx1, ly1 = obs_xs[left_mask][-2], obs_ys[left_mask][-2]
                        slope    = (ly2 - ly1) / (lx2 - lx1 + 1e-10)
                        slope    = min(slope, abs(ly2 / (lx2 + 1e-10)) * 1.5)
                    else:
                        slope = 0
                    pred = ly2 + slope * (tgt_m - lx2)

                else:
                    rx2, ry2 = obs_xs[right_mask][0], obs_ys[right_mask][0]
                    if right_mask.sum() >= 2:
                        rx1, ry1 = obs_xs[right_mask][1], obs_ys[right_mask][1]
                        slope    = (ry1 - ry2) / (rx1 - rx2 + 1e-10)
                        slope    = min(slope, abs(ry2 / (abs(rx2) + 1e-10)) * 1.5)
                    else:
                        slope = 0
                    pred = ry2 + slope * (tgt_m - rx2)

                filled_cross.loc[ti, cols_grp[mi]] = max(pred, 0.005)

    filled_cross = filled_cross.ffill().bfill()

    # ── Pass 2: PCHIP temporal fill (expiry rows only) ────────────────────
    filled_temp    = df[iv_cols].copy()
    expiry_indices = np.where(expiry_mask)[0]

    for col in iv_cols:
        series        = df[col].copy()
        series_expiry = series.iloc[expiry_indices]
        obs_mask      = ~series_expiry.isna()
        ox            = expiry_indices[obs_mask.values]
        oy            = series_expiry[obs_mask].values
        miss_x        = expiry_indices[~obs_mask.values]
        if len(miss_x) == 0 or len(ox) < 2:
            continue
        pi = PchipInterpolator(ox, oy, extrapolate=True)
        filled_temp.loc[miss_x, col] = np.clip(pi(miss_x), 0.005, None)

    filled_temp = filled_temp.ffill().bfill()

    # ── Blend (expiry rows only) ──────────────────────────────────────────
    final     = df[iv_cols].copy()
    miss_orig = df[iv_cols].isna()

    for col in iv_cols:
        # fix: use numpy boolean arrays to avoid pandas index alignment issues
        miss = miss_orig[col].values & expiry_mask
        final.loc[miss, col] = (
            (1 - ALPHA) * filled_cross.loc[miss, col].values
            + ALPHA     * filled_temp.loc[miss, col].values
        )

    final = final.clip(lower=0.005)
    return final, expiry_mask


# alias so main() can call friends_fill_wide consistently
friends_fill_wide = fill_wide


# ─────────────────────────────────────────────
# 3. MLP
# ─────────────────────────────────────────────
class IVSurfaceMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=None, dropout=0.1):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [128, 64, 32]
        layers = []
        prev   = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(X_train, y_train, X_val, y_val,
              epochs=EPOCHS, batch_size=BATCH_SIZE, patience=PATIENCE):
    scaler      = StandardScaler()
    X_train_trf = scaler.fit_transform(X_train)
    X_val_trf   = scaler.transform(X_val)

    Xtt = torch.tensor(X_train_trf, dtype=torch.float32).to(DEVICE)
    ytt = torch.tensor(y_train,     dtype=torch.float32).to(DEVICE)
    Xvt = torch.tensor(X_val_trf,   dtype=torch.float32).to(DEVICE)
    yvt = torch.tensor(y_val,       dtype=torch.float32).to(DEVICE)

    loader  = DataLoader(TensorDataset(Xtt, ytt), batch_size=batch_size, shuffle=True)
    model   = IVSurfaceMLP(X_train.shape[1]).to(DEVICE)
    opt     = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.MSELoss()

    best_loss, best_state, no_imp = float('inf'), None, 0
    best_epoch = 0
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item()
        sched.step()

        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(Xvt), yvt).item()

        train_losses.append(epoch_loss / len(loader))
        val_losses.append(vl)

        if vl < best_loss:
            best_loss  = vl
            best_epoch = epoch          # fix: track exact best epoch
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp     = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    model.load_state_dict(best_state)
    return model, scaler, best_epoch, np.array(train_losses), np.array(val_losses)


def predict_mlp(model, scaler, X):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(scaler.transform(X), dtype=torch.float32).to(DEVICE)
        return model(X_t).cpu().numpy()


# ─────────────────────────────────────────────
# 4. FEATURE ENGINEERING  (vectorized, no iterrows)
# ─────────────────────────────────────────────
def build_long_features(df_wide, ce_cols, pe_cols):
    """
    Convert wide IV dataframe to long format and engineer features.
    Fully vectorized — no iterrows.
    """
    frames = []
    for opt_type, cols_grp in [('CE', ce_cols), ('PE', pe_cols)]:
        melted = df_wide[['datetime', 'underlying_price'] + cols_grp].melt(
            id_vars=['datetime', 'underlying_price'],
            value_vars=cols_grp,
            var_name='col_name',
            value_name='iv'
        )
        melted['strike']   = melted['col_name'].map(get_strike)
        melted['opt_type'] = opt_type
        melted['is_ce']    = 1.0 if opt_type == 'CE' else 0.0
        melted['col_idx']  = melted['col_name'].map({c: i for i, c in enumerate(cols_grp)})
        frames.append(melted)

    long_df = pd.concat(frames, ignore_index=True)

    long_df['moneyness']         = long_df['strike'] / long_df['underlying_price']
    long_df['log_moneyness']     = np.log(long_df['moneyness'])
    long_df['dist_from_atm_pct'] = (
        (long_df['strike'] - long_df['underlying_price']).abs()
        / long_df['underlying_price'] * 100
    )

    # neighbor IVs via sort + groupby shift (vectorized, replaces per-row logic)
    long_df = long_df.sort_values(['datetime', 'opt_type', 'strike']).reset_index(drop=True)
    grp = long_df.groupby(['datetime', 'opt_type'])['iv']
    long_df['iv_neighbor_+1'] = grp.shift(-1)
    long_df['iv_neighbor_-1'] = grp.shift(+1)
    long_df['iv_neighbor_+2'] = grp.shift(-2)
    long_df['iv_neighbor_-2'] = grp.shift(+2)

    long_df['iv_neighbor_mean'] = long_df[
        ['iv_neighbor_-1', 'iv_neighbor_+1']
    ].mean(axis=1)

    long_df['wide_iv_neighbor_mean'] = long_df[
        ['iv_neighbor_-2', 'iv_neighbor_-1', 'iv_neighbor_+1', 'iv_neighbor_+2']
    ].mean(axis=1)

    # temporal rolling means — shift(1) prevents leakage
    long_df = long_df.sort_values(['strike', 'opt_type', 'datetime']).reset_index(drop=True)
    for window, col in [(5, 'iv_roll_mean_5'), (10, 'iv_roll_mean_10')]:
        long_df[col] = long_df.groupby(['strike', 'opt_type'])['iv'].transform(
            lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()
        )

    # fix: for missing-iv rows, rolling mean is NaN — fill with group median
    # (filling with iv itself doesn't help when iv is also NaN)
    group_med = long_df.groupby(['strike', 'opt_type'])['iv'].transform('median')
    long_df['iv_roll_mean_5']  = long_df['iv_roll_mean_5'].fillna(group_med)
    long_df['iv_roll_mean_10'] = long_df['iv_roll_mean_10'].fillna(group_med)

    return long_df


# ─────────────────────────────────────────────
# 5. MAIN
# ─────────────────────────────────────────────
def main():
    print("Loading data...")
    df_raw           = load_raw()
    ce_cols, pe_cols = get_option_cols(df_raw)
    iv_cols          = ce_cols + pe_cols

    dates       = df_raw['datetime'].dt.date
    all_dates   = sorted(dates.unique())
    expiry_date = all_dates[-1]                          # last calendar day only
    expiry_mask = (dates == expiry_date).values

    print(f"Total rows: {len(df_raw)}  |  Expiry rows: {expiry_mask.sum()}")
    print(f"Device: {DEVICE}")

    # ── A. Expiry day fill (cross-sectional + PCHIP) ───────────────────────
    print("\nRunning cross-sectional + PCHIP fill for expiry day...")
    friends_filled, _ = friends_fill_wide(df_raw, ce_cols, pe_cols)

    # ── B. Build long-format features on NON-EXPIRY rows ──────────────────
    print("Building long-format features (non-expiry)...")
    df_non_expiry = df_raw[~expiry_mask].copy().reset_index(drop=True)
    long_df       = build_long_features(df_non_expiry, ce_cols, pe_cols)

    # ── C. Train MLP on ALL observed non-expiry data ───────────────────────
    print("Training MLP on all observed non-expiry data...")
    obs_all = long_df[long_df['iv'].notna()]
    X_train = obs_all[FEATURE_COLS].fillna(0).values
    y_train = obs_all['iv'].values

    # 90/10 split purely for early-stopping signal — not used for evaluation
    split = int(0.9 * len(X_train))
    model, scaler, best_ep, _, _ = train_mlp(
        X_train[:split], y_train[:split],
        X_train[split:], y_train[split:]
    )
    print(f"  Training done. best_epoch={best_ep}")

    # ── D. Predict missing non-expiry cells ───────────────────────────────
    print("Building full long-format for prediction...")
    long_full = build_long_features(df_raw, ce_cols, pe_cols)

    long_miss = long_full[
        long_full['iv'].isna() &
        (long_full['datetime'].dt.date != expiry_date)
    ].copy()

    print(f"  MLP will fill {len(long_miss)} missing non-expiry cells.")

    if len(long_miss) > 0:
        X_miss               = long_miss[FEATURE_COLS].fillna(0).values
        long_miss['iv_pred'] = predict_mlp(model, scaler, X_miss).clip(min=0.005)
    else:
        long_miss['iv_pred'] = pd.Series(dtype=float)

    # ── E. Assemble final wide dataframe ──────────────────────────────────
    print("Assembling final wide dataframe...")
    final_wide = df_raw[['datetime', 'underlying_price'] + iv_cols].copy()
    miss_orig  = df_raw[iv_cols].isna()

    # E1. Write MLP predictions via vectorized merge (no slow for-loop)
    if len(long_miss) > 0:
        # reconstruct col_name from strike + opt_type
        long_miss['col_name'] = (
            'NIFTY27JAN26'
            + long_miss['strike'].astype(int).astype(str)
            + long_miss['opt_type']
        )

        # fix: assert lookup integrity before writing
        assert long_miss['col_name'].isin(iv_cols).all(), \
            "Some predicted col_names don't match iv_cols — check get_strike / naming"

        for col in iv_cols:
            sub = (long_miss[long_miss['col_name'] == col]
                   [['datetime', 'iv_pred']]
                   .drop_duplicates('datetime'))
            if sub.empty:
                continue
            # merge predictions aligned by datetime
            merged = final_wide[['datetime']].merge(sub, on='datetime', how='left')
            mask   = miss_orig[col].values & ~expiry_mask
            final_wide.loc[mask, col] = merged.loc[mask, 'iv_pred'].values

    # E2. Overwrite expiry missing cells with cross-sectional+PCHIP fills
    for col in iv_cols:
        mask = miss_orig[col].values & expiry_mask
        if mask.any():
            final_wide.loc[mask, col] = friends_filled.loc[mask, col].values

    # ── F. Safety ffill/bfill for any remaining NaN ───────────────────────
    still_missing = final_wide[iv_cols].isna().sum().sum()
    if still_missing:
        print(f"  {still_missing} cells still NaN after both fills — applying ffill/bfill")
        final_wide[iv_cols] = final_wide[iv_cols].ffill().bfill()

    final_wide[iv_cols] = final_wide[iv_cols].clip(lower=0.005)

    assert final_wide[iv_cols].isnull().sum().sum() == 0, "NaNs remain after all fills!"
    print(f"IV range: {final_wide[iv_cols].min().min():.4f} – {final_wide[iv_cols].max().max():.4f}")

    # ── G. Export filled wide CSV ──────────────────────────────────────────
    print(f"\nSaving filled wide CSV → {OUTPUT_CSV}")
    # use a copy so df_raw['datetime'] stays as Timestamp for step H
    out_wide = final_wide.copy()
    out_wide['datetime'] = df_raw['datetime'].dt.strftime('%d-%m-%Y %H:%M')
    out_wide[['datetime', 'underlying_price'] + iv_cols].to_csv(OUTPUT_CSV, index=False)

    # ── H. Build Kaggle submission CSV ─────────────────────────────────────
    print(f"Building submission CSV → {SUBMIT_CSV}")
    original = pd.read_csv(RAW_CSV)
    filled   = pd.read_csv(OUTPUT_CSV)

    rows_out = []
    sub_cols = [c for c in original.columns if c not in ('datetime', 'underlying_price')]
    for col in sub_cols:
        nan_idx = original.index[original[col].isna()]
        for idx in nan_idx:
            rows_out.append({
                'id':    f"{original.loc[idx, 'datetime']}||{col}",
                'value': filled.loc[idx, col]
            })

    (pd.DataFrame(rows_out, columns=['id', 'value'])
       .sort_values('id')
       .reset_index(drop=True)
       .to_csv(SUBMIT_CSV, index=False))

    print(f"\nDone! Submission saved → {SUBMIT_CSV}")


if __name__ == '__main__':
    main()

Loading data...
Total rows: 975  |  Expiry rows: 75
Device: cpu

Running cross-sectional + PCHIP fill for expiry day...
Building long-format features (non-expiry)...
Training MLP on all observed non-expiry data...
  Training done. best_epoch=158
Building full long-format for prediction...
  MLP will fill 5052 missing non-expiry cells.
Assembling final wide dataframe...
IV range: 0.0168 – 5.4970

Saving filled wide CSV → ../outputs/2_submissions/filled_dataset.csv
Building submission CSV → ../outputs/2_submissions/submission.csv

Done! Submission saved → ../outputs/2_submissions/submission.csv


In [10]:
import numpy as np
import pandas as pd

df1 = pd.read_csv('../data/raw/submission_1.csv')
df2 = pd.read_csv('../data/raw/submission_2.csv')

print(df1.head())
print(df2.head())

                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.163295
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.117317
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.100722
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.170281
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.159527
                                            id,value
0  07-01-2026 09:15||NIFTY27JAN2624100PE,0.163295077
1  07-01-2026 09:15||NIFTY27JAN2625500CE,0.117316511
2  07-01-2026 09:15||NIFTY27JAN2625800CE,0.100722307
3  07-01-2026 09:20||NIFTY27JAN2624000PE,0.170280871
4  07-01-2026 09:20||NIFTY27JAN2624200PE,0.159526688


In [12]:
df2 = pd.read_csv('../data/raw/submission_2.csv')

# Split the single column into two columns
df2[['id', 'value']] = df2['id,value'].str.split(',', expand=True)

df2['value'] = df2['value'].astype(float)

df2 = df2[['id', 'value']]

In [13]:
merged = df1.merge(
    df2,
    on='id',
    how='outer',
    suffixes=('_df1', '_df2')
)

mismatch = merged[
    ~np.isclose(
        merged['value_df1'],
        merged['value_df2'],
        rtol=0,
        atol=1e-12
    )
]

print(f"Total mismatches: {len(mismatch)}")

mismatch[['id', 'value_df1', 'value_df2']]

Total mismatches: 5444


,id,value_df1,value_df2
0,07-01-2026 09:15||NIFTY27JAN2624100PE,0.163295,0.163295
1,07-01-2026 09:15||NIFTY27JAN2625500CE,0.117317,0.117317
2,07-01-2026 09:15||NIFTY27JAN2625800CE,0.100722,0.100722
3,07-01-2026 09:20||NIFTY27JAN2624000PE,0.170281,0.170281
4,07-01-2026 09:20||NIFTY27JAN2624200PE,0.159527,0.159527
...,...,...,...
5455,27-01-2026 15:25||NIFTY27JAN2625100PE,0.678919,0.678919
5456,27-01-2026 15:25||NIFTY27JAN2626100CE,3.377353,3.377353
5457,27-01-2026 15:25||NIFTY27JAN2626300CE,4.036605,3.936605
5458,27-01-2026 15:25||NIFTY27JAN2626400CE,4.341582,4.241582


In [14]:
merged['diff'] = abs(
    merged['value_df1'] - merged['value_df2']
)

mismatch = merged[
    merged['diff'] > 0.01
]

print("Total mismatches:", len(mismatch))

display(
    mismatch[['id', 'value_df1', 'value_df2', 'diff']]
)

Total mismatches: 8


,id,value_df1,value_df2,diff
5408,27-01-2026 14:45||NIFTY27JAN2624000PE,1.623908,1.523908,0.1
5424,27-01-2026 14:55||NIFTY27JAN2625800CE,1.152414,1.452414,0.3
5432,27-01-2026 15:05||NIFTY27JAN2624200PE,1.848016,1.948016,0.1
5440,27-01-2026 15:10||NIFTY27JAN2625800CE,1.352536,1.252536,0.1
5445,27-01-2026 15:15||NIFTY27JAN2626400CE,2.704041,2.904041,0.2
5457,27-01-2026 15:25||NIFTY27JAN2626300CE,4.036605,3.936605,0.1
5458,27-01-2026 15:25||NIFTY27JAN2626400CE,4.341582,4.241582,0.1
5459,27-01-2026 15:25||NIFTY27JAN2626500CE,4.802666,4.902666,0.1
